In [1]:
"""
====================================================================
TISSUE-STRUCTURE RECONSTRUCTION PRETRAINING -- v2 (Swin-Tiny)
Cross-Centre Generalization in Histopathology

Swin-Tiny: hierarchical window-based attention, 768-dim features.
Requires 224px input (window partitioning constraint).
Memory-heavy — uses batch 8 with grad accum 6 (effective 48).

Run:
    python reconstruction_swin_tiny.py --check
    python reconstruction_swin_tiny.py --smoke
    python reconstruction_swin_tiny.py
====================================================================
"""

import os
import sys
import math
import random
import pickle
import gc
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG  --  Swin-Tiny
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CACHE_DIR = "/kaggle/working/cache"
    RESULTS_DIR = "/kaggle/working/results"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"

    # --- task ---
    GRID_SIZE = 3
    NUM_POSITIONS = 9
    NUM_PERMUTATIONS = 500
    MASK_PROB = 0.15
    MAX_MASKED = 2

    # --- data (reduced — Swin is memory-heavy) ---
    TRAIN_SAMPLES_PER_EPOCH = 10000
    VAL_SAMPLES = 2000
    VAL_PATIENT_FRAC = 0.25

    # --- model ---
    BACKBONE = 'swin_tiny'
    CNN_INPUT_SIZE = 96
    VIT_INPUT_SIZE = 224
    PRETRAINED = True
    DROPOUT = 0.1                     # lower, like ViT
    FREEZE_STAGES = 0
    CONTEXT_LAYERS = 3
    BACKBONE_LR_MULT = 0.1            # transformer — lower backbone LR

    # --- training (OOM-safe) ---
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 6              # effective batch 48
    VAL_BATCH_SIZE = 8
    LR = 3e-4
    WARMUP_EPOCHS = 4                 # longer for transformer
    WEIGHT_DECAY = 0.05
    EPOCHS = 60
    AUX_LOSS_WEIGHT = 1.0
    LABEL_SMOOTHING = 0.05
    GRAD_CLIP = 1.0
    EARLY_STOP_PATIENCE = 12
    NUM_WORKERS = 4
    SEED = 42
    USE_AMP = True
    RESUME = True

    ALL_CENTERS = [0, 1, 2, 3, 4]
    TRAIN_CENTERS = [0, 1, 3, 4]
    HELD_OUT_CENTER = 2

    SMOKE_EPOCHS = 3
    SMOKE_TRAIN = 3000
    SMOKE_VAL = 1000


def get_input_size(backbone):
    if backbone in ('vit_small', 'vit_tiny', 'swin_tiny', 'swin_small'):
        return Config.VIT_INPUT_SIZE
    return Config.CNN_INPUT_SIZE


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed); random.seed(seed)


for _d in (Config.CACHE_DIR, Config.RESULTS_DIR, Config.CHECKPOINT_DIR):
    os.makedirs(_d, exist_ok=True)


# ====================================================================
# DATA LOADING
# ====================================================================
REQUIRED = ['patient', 'node', 'center', 'x_coord', 'y_coord']


def load_full_dataset():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    print(f"Splits: {', '.join(f'{k}={len(ds[k]):,}' for k in ds.keys())}")
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ Concatenated: {len(full):,} patches")
    return full


def check_schema(dataset):
    print("\n" + "=" * 60)
    print("SCHEMA CHECK")
    print("=" * 60)
    for name, feat in dataset.features.items():
        print(f"  {name:<12} {feat}")
    missing = [c for c in REQUIRED if c not in dataset.features]
    if missing:
        print(f"\n❌ missing: {missing}")
        return False
    print(f"\n✓ required columns present, {len(dataset):,} patches")
    return True


# ====================================================================
# SPATIAL INDEX
# ====================================================================
class Lattice:
    __slots__ = ('patient', 'node', 'center', 'idx_grid', 'valid_centers')

    def __init__(self, patient, node, center, patches, grid_size):
        self.patient, self.node, self.center = patient, node, center
        xs = sorted({p['x'] for p in patches})
        ys = sorted({p['y'] for p in patches})
        x_to_c = {x: i for i, x in enumerate(xs)}
        y_to_r = {y: i for i, y in enumerate(ys)}
        g = np.full((len(ys), len(xs)), -1, dtype=np.int64)
        for p in patches:
            g[y_to_r[p['y']], x_to_c[p['x']]] = p['idx']
        self.idx_grid = g
        half = grid_size // 2
        H, W = g.shape
        valid = []
        for r in range(half, H - half):
            for c in range(half, W - half):
                win = g[r - half:r + half + 1, c - half:c + half + 1]
                if (win >= 0).all():
                    valid.append((r, c))
        self.valid_centers = np.array(valid, dtype=np.int32) if valid else np.zeros((0, 2), dtype=np.int32)

    def window(self, r, c, grid_size):
        half = grid_size // 2
        return self.idx_grid[r - half:r + half + 1, c - half:c + half + 1].ravel()


class SpatialIndex:
    def __init__(self, dataset, grid_size, cache_path=None):
        self.grid_size = grid_size
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, 'rb') as f:
                self.lattices = pickle.load(f)
            print(f"✓ loaded {len(self.lattices)} lattices from cache")
        else:
            self.lattices = self._build(dataset)
            if cache_path:
                with open(cache_path, 'wb') as f:
                    pickle.dump(self.lattices, f)
                print(f"✓ cached → {cache_path}")
        self.report()

    def _build(self, dataset):
        print("Building lattices...")
        meta = dataset.remove_columns(['image']).to_pandas()
        groups = defaultdict(list)
        for row in tqdm(meta.itertuples(index=True), total=len(meta), desc="index"):
            groups[(int(row.patient), int(row.node), int(row.center))].append(
                {'x': int(row.x_coord), 'y': int(row.y_coord), 'idx': int(row.Index)})
        lattices = []
        for (pat, node, ctr), patches in tqdm(groups.items(), desc="lattices"):
            lat = Lattice(pat, node, ctr, patches, self.grid_size)
            if len(lat.valid_centers):
                lattices.append(lat)
        return lattices

    def report(self):
        by_ctr = defaultdict(int)
        pats_by_ctr = defaultdict(set)
        for lat in self.lattices:
            by_ctr[lat.center] += len(lat.valid_centers)
            pats_by_ctr[lat.center].add(lat.patient)
        print("\nValid grid positions per centre:")
        for c in sorted(by_ctr):
            print(f"  centre {c}: {by_ctr[c]:>9,} positions   {len(pats_by_ctr[c])} patients")
        print(f"  TOTAL   : {sum(by_ctr.values()):>9,} positions")

    def patients_in(self, centers):
        return sorted({lat.patient for lat in self.lattices if lat.center in centers})

    def subset(self, centers, patients):
        pats = set(patients)
        return [lat for lat in self.lattices if lat.center in centers and lat.patient in pats]


# ====================================================================
# PERMUTATION GENERATOR
# ====================================================================
class PermutationGenerator:
    def __init__(self, num_permutations=500, num_positions=9, seed=1234):
        self.num_positions = num_positions
        rng = np.random.RandomState(seed)
        pool = set()
        while len(pool) < max(num_permutations * 20, 5000):
            pool.add(tuple(rng.permutation(num_positions)))
        pool = np.array(sorted(pool))
        sel = [0]
        mind = (pool != pool[0]).sum(1).astype(np.int32)
        for _ in range(num_permutations - 1):
            mind[sel] = -1
            nxt = int(np.argmax(mind))
            sel.append(nxt)
            mind = np.minimum(mind, (pool != pool[nxt]).sum(1))
        self.permutations = [tuple(int(v) for v in pool[i]) for i in sel]

    def sample(self):
        i = random.randrange(len(self.permutations))
        return self.permutations[i], i

    def __getitem__(self, i): return self.permutations[i]
    def __len__(self): return len(self.permutations)


# ====================================================================
# DATASET
# ====================================================================
class DynamicGridDataset(Dataset):
    def __init__(self, hf_dataset, lattices, perms, transform,
                 length, grid_size=3, train=True, seed=7,
                 mask_prob=0.0, max_masked=2):
        self.ds = hf_dataset
        self.lattices = lattices
        self.perms = perms
        self.transform = transform
        self.length = length
        self.grid_size = grid_size
        self.train = train
        self.mask_prob = mask_prob if train else 0.0
        self.max_masked = max_masked
        counts = np.array([len(l.valid_centers) for l in lattices], dtype=np.float64)
        if counts.sum() == 0:
            raise RuntimeError("No valid grid positions in this subset.")
        self.p = counts / counts.sum()
        if not train:
            rng = np.random.RandomState(seed)
            self.fixed = []
            for _ in range(length):
                li = int(rng.choice(len(lattices), p=self.p))
                lat = lattices[li]
                ci = int(rng.randint(len(lat.valid_centers)))
                pi = int(rng.randint(len(perms)))
                self.fixed.append((li, ci, pi))

    def __len__(self):
        return self.length

    def _load(self, idx):
        img = self.ds[int(idx)]['image']
        return img.convert('RGB') if img.mode != 'RGB' else img

    def __getitem__(self, i):
        if self.train:
            li = int(np.random.choice(len(self.lattices), p=self.p))
            lat = self.lattices[li]
            ci = random.randrange(len(lat.valid_centers))
            perm, perm_idx = self.perms.sample()
        else:
            li, ci, perm_idx = self.fixed[i]
            lat = self.lattices[li]
            perm = self.perms[perm_idx]
        r, c = lat.valid_centers[ci]
        flat = lat.window(int(r), int(c), self.grid_size)
        tiles = torch.stack([self.transform(self._load(k)) for k in flat])
        shuffled = tiles[list(perm)]
        mask = torch.zeros(self.grid_size ** 2, dtype=torch.bool)
        if self.mask_prob > 0:
            n = min(int(np.random.binomial(self.grid_size ** 2, self.mask_prob)), self.max_masked)
            if n:
                slots = np.random.choice(self.grid_size ** 2, n, replace=False)
                shuffled[slots] = 0.0
                mask[slots] = True
        return {
            'images': shuffled,
            'perm_idx': perm_idx,
            'perm': torch.tensor(perm, dtype=torch.long),
            'mask': mask,
            'center': lat.center,
            'patient': lat.patient,
        }


# ====================================================================
# MODEL  --  Swin-Tiny
# ====================================================================
class ReconstructionModel(nn.Module):
    def __init__(self, backbone_name='swin_tiny', num_positions=9,
                 num_permutations=500, pretrained=True, dropout=0.1,
                 freeze_stages=0, context_layers=3):
        super().__init__()
        self.backbone_name = backbone_name
        self.num_positions = num_positions
        self.num_permutations = num_permutations
        self.input_size = get_input_size(backbone_name)
        self._build_backbone(backbone_name, pretrained, freeze_stages)
        assert self.feature_dim % 8 == 0
        self.proj = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.LayerNorm(self.feature_dim),
        )
        self.context = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=self.feature_dim, nhead=8,
                dim_feedforward=min(4 * self.feature_dim, 2048),
                dropout=dropout, batch_first=True, norm_first=True),
            num_layers=context_layers)
        self.slot_embed = nn.Parameter(torch.zeros(1, num_positions, self.feature_dim))
        nn.init.trunc_normal_(self.slot_embed, std=0.02)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, self.feature_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.perm_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim * num_positions, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_permutations))
        self.position_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_positions))

    def _build_backbone(self, name, pretrained, freeze_stages):
        if name == 'swin_tiny':
            import timm
            self.backbone = timm.create_model(
                'swin_tiny_patch4_window7_224',
                pretrained=pretrained, num_classes=0)
            self.feature_dim = 768
            self.pool = None
            self._frozen_params = 0
        else:
            raise ValueError(f"Backbone {name} not supported in this file")

    def _encode(self, x, mask=None):
        B, N = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        f = self.proj(f).view(B, N, self.feature_dim)
        if mask is not None and mask.any():
            f = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(f), f)
        return self.context(f + self.slot_embed)

    def forward(self, x, mask=None):
        ctx = self._encode(x, mask)
        return self.perm_head(ctx.flatten(1)), self.position_head(ctx)

    @torch.no_grad()
    def get_embedding(self, x):
        return self._encode(x)

    def get_param_groups(self, base_lr, backbone_mult=0.1):
        backbone_params = list(self.backbone.parameters())
        head_params = [p for n, p in self.named_parameters()
                       if not n.startswith('backbone.') and p.requires_grad]
        return [
            {'params': backbone_params, 'lr': base_lr * backbone_mult},
            {'params': head_params, 'lr': base_lr},
        ]


# ====================================================================
# TRAINER
# ====================================================================
def _scaler(enabled):
    try: return torch.amp.GradScaler('cuda', enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)


def _autocast(enabled):
    try: return torch.amp.autocast('cuda', enabled=enabled)
    except Exception: return torch.cuda.amp.autocast(enabled=enabled)


class Trainer:
    def __init__(self, model, device, cfg, steps_per_epoch):
        self.model, self.device, self.cfg = model, device, cfg
        param_groups = model.get_param_groups(cfg.LR, cfg.BACKBONE_LR_MULT)
        self.optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.WEIGHT_DECAY)
        total = cfg.EPOCHS * steps_per_epoch
        warm = cfg.WARMUP_EPOCHS * steps_per_epoch
        def lr_lambda(step):
            if step < warm:
                return step / max(1, warm)
            prog = (step - warm) / max(1, total - warm)
            return 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda)
        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
        self.use_amp = cfg.USE_AMP and device.type == 'cuda'
        self.scaler = _scaler(self.use_amp)
        self.history = defaultdict(list)
        self.start_epoch = 0
        self.best_pos = 0.0
        self.bad_epochs = 0

    def save(self, epoch, is_best):
        ck = {'epoch': epoch, 'model_state_dict': self.model.state_dict(),
              'optimizer_state_dict': self.optimizer.state_dict(),
              'scheduler_state_dict': self.scheduler.state_dict(),
              'scaler_state_dict': self.scaler.state_dict(),
              'best_pos': self.best_pos, 'bad_epochs': self.bad_epochs,
              'history': dict(self.history), 'backbone': self.model.backbone_name}
        torch.save(ck, f"{self.cfg.CHECKPOINT_DIR}/v2_swin_tiny_latest.pt")
        if is_best:
            torch.save(ck, f"{self.cfg.CHECKPOINT_DIR}/v2_swin_tiny_best.pt")

    def load(self, path):
        ck = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ck['model_state_dict'])
        try:
            self.optimizer.load_state_dict(ck['optimizer_state_dict'])
            self.scheduler.load_state_dict(ck['scheduler_state_dict'])
            self.scaler.load_state_dict(ck['scaler_state_dict'])
        except Exception as e:
            print(f"⚠ optimizer state partial: {e}")
        self.start_epoch = ck['epoch'] + 1
        self.best_pos = ck['best_pos']
        self.bad_epochs = ck.get('bad_epochs', 0)
        self.history = defaultdict(list, ck['history'])
        print(f"✓ resumed from epoch {self.start_epoch}, best pos {self.best_pos:.4f}")

    def _forward(self, batch):
        x = batch['images'].to(self.device, non_blocking=True)
        m = batch['mask'].to(self.device, non_blocking=True)
        perm_idx = batch['perm_idx'].to(self.device, non_blocking=True)
        perm = batch['perm'].to(self.device, non_blocking=True)
        perm_logits, pos_logits = self.model(x, m)
        l_perm = self.criterion(perm_logits, perm_idx)
        l_pos = self.criterion(pos_logits.reshape(-1, self.model.num_positions), perm.reshape(-1))
        loss = l_perm + self.cfg.AUX_LOSS_WEIGHT * l_pos
        with torch.no_grad():
            pc = (perm_logits.argmax(1) == perm_idx).sum().item()
            vis = ~m
            oc = ((pos_logits.argmax(-1) == perm) & vis).sum().item()
            nvis = vis.sum().item()
        return loss, l_perm, l_pos, pc, oc, perm_idx.size(0), nvis

    def train_epoch(self, loader):
        self.model.train()
        tl = 0.0; pc = oc = ns = nv = 0
        accum = self.cfg.GRAD_ACCUM_STEPS
        self.optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(loader, desc="train", leave=False)
        for step, batch in enumerate(pbar):
            with _autocast(self.use_amp):
                loss, lp, lo, c, o, bs, nvis = self._forward(batch)
                loss = loss / accum
            self.scaler.scale(loss).backward()
            if (step + 1) % accum == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in self.model.parameters() if p.requires_grad], self.cfg.GRAD_CLIP)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad(set_to_none=True)
                self.scheduler.step()
            tl += loss.item() * accum * bs
            pc += c; oc += o; ns += bs; nv += nvis
            pbar.set_postfix({'loss': f'{tl/ns:.3f}', 'pos': f'{oc/nv:.3f}'})
            del batch, loss
            if step % 20 == 0:
                torch.cuda.empty_cache()
        if len(loader) % accum != 0:
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in self.model.parameters() if p.requires_grad], self.cfg.GRAD_CLIP)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad(set_to_none=True)
            self.scheduler.step()
        self.history['train_loss'].append(tl / ns)
        self.history['train_perm'].append(pc / ns)
        self.history['train_pos'].append(oc / nv)
        torch.cuda.empty_cache()
        return tl / ns, pc / ns, oc / nv

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        tl = 0.0; pc = oc = ns = nv = 0
        for batch in tqdm(loader, desc="val", leave=False):
            with _autocast(self.use_amp):
                loss, lp, lo, c, o, bs, nvis = self._forward(batch)
            tl += loss.item() * bs; pc += c; oc += o; ns += bs; nv += nvis
            del batch, loss
        self.history['val_loss'].append(tl / ns)
        self.history['val_perm'].append(pc / ns)
        self.history['val_pos'].append(oc / nv)
        torch.cuda.empty_cache()
        return tl / ns, pc / ns, oc / nv

    def fit(self, train_loader, val_loader):
        latest = f"{self.cfg.CHECKPOINT_DIR}/v2_swin_tiny_latest.pt"
        if self.cfg.RESUME and os.path.exists(latest):
            self.load(latest)
        ch_perm = 100.0 / self.cfg.NUM_PERMUTATIONS
        ch_pos = 100.0 / self.cfg.NUM_POSITIONS
        print(f"\nchance: perm {ch_perm:.2f}% | per-patch {ch_pos:.2f}%")
        print(f"effective batch: {self.cfg.BATCH_SIZE} × {self.cfg.GRAD_ACCUM_STEPS} = {self.cfg.BATCH_SIZE * self.cfg.GRAD_ACCUM_STEPS}\n")
        for ep in range(self.start_epoch, self.cfg.EPOCHS):
            lr = self.scheduler.get_last_lr()[0]
            print(f"Epoch {ep+1}/{self.cfg.EPOCHS}   lr={lr:.2e}")
            trl, trp, tro = self.train_epoch(train_loader)
            val, vap, vao = self.validate(val_loader)
            gap = tro - vao
            print(f"  train  loss {trl:.4f}   perm {trp*100:6.2f}%   pos {tro*100:6.2f}%")
            print(f"  val    loss {val:.4f}   perm {vap*100:6.2f}%   pos {vao*100:6.2f}%")
            print(f"  per-patch above chance: {vao*100 - ch_pos:+.2f} pp   | gap: {gap*100:.1f} pp")
            is_best = vao > self.best_pos + 1e-4
            if is_best:
                self.best_pos = vao
                self.bad_epochs = 0
                print(f"  ✓ new best per-patch {vao*100:.2f}%")
            else:
                self.bad_epochs += 1
            self.save(ep, is_best)
            if self.bad_epochs >= self.cfg.EARLY_STOP_PATIENCE:
                print(f"\n⏹ early stop after {self.cfg.EARLY_STOP_PATIENCE} epochs")
                break
        return self.history


def build_transforms(backbone):
    size = get_input_size(backbone)
    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    train = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.15),
        transforms.ToTensor(), norm,
    ])
    val = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(), norm])
    return train, val


def create_dataloaders(hf_dataset, index, cfg):
    patients = index.patients_in(cfg.TRAIN_CENTERS)
    rng = random.Random(cfg.SEED)
    rng.shuffle(patients)
    n_val = max(2, int(round(cfg.VAL_PATIENT_FRAC * len(patients))))
    val_patients, train_patients = patients[:n_val], patients[n_val:]
    train_lat = index.subset(cfg.TRAIN_CENTERS, train_patients)
    val_lat = index.subset(cfg.TRAIN_CENTERS, val_patients)
    print(f"\nPatients: {len(train_patients)} train / {len(val_patients)} val")
    print(f"Positions: {sum(len(l.valid_centers) for l in train_lat):,} train / "
          f"{sum(len(l.valid_centers) for l in val_lat):,} val")
    if not train_lat or not val_lat:
        raise RuntimeError("Empty train or val subset.")
    ttf, vtf = build_transforms(cfg.BACKBONE)
    perms = PermutationGenerator(cfg.NUM_PERMUTATIONS, cfg.NUM_POSITIONS)
    train_ds = DynamicGridDataset(hf_dataset, train_lat, perms, ttf,
                                  length=cfg.TRAIN_SAMPLES_PER_EPOCH,
                                  grid_size=cfg.GRID_SIZE, train=True,
                                  mask_prob=cfg.MASK_PROB, max_masked=cfg.MAX_MASKED)
    val_ds = DynamicGridDataset(hf_dataset, val_lat, perms, vtf,
                                length=cfg.VAL_SAMPLES,
                                grid_size=cfg.GRID_SIZE, train=False)
    train_common = dict(batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS,
                       pin_memory=True, worker_init_fn=worker_init_fn,
                       persistent_workers=cfg.NUM_WORKERS > 0)
    val_common = dict(batch_size=cfg.VAL_BATCH_SIZE, num_workers=cfg.NUM_WORKERS,
                     pin_memory=True, worker_init_fn=worker_init_fn,
                     persistent_workers=cfg.NUM_WORKERS > 0)
    return (DataLoader(train_ds, shuffle=False, drop_last=True, **train_common),
            DataLoader(val_ds, shuffle=False, **val_common))


def plot_history(hist, path=None):
    path = path or f"{Config.RESULTS_DIR}/v2_swin_tiny_curves.png"
    ep = range(1, len(hist['train_loss']) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    ax[0].plot(ep, hist['train_loss'], label='train')
    ax[0].plot(ep, hist['val_loss'], label='val')
    ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, np.array(hist['train_pos'])*100, label='train')
    ax[1].plot(ep, np.array(hist['val_pos'])*100, label='val')
    ax[1].axhline(100/Config.NUM_POSITIONS, ls='--', c='grey', label='chance')
    ax[1].axhline(30, ls=':', c='green', label='target 30%')
    ax[1].set_title('Per-patch accuracy (%)'); ax[1].legend(); ax[1].grid(alpha=.3)
    ax[2].plot(ep, np.array(hist['train_perm'])*100, label='train')
    ax[2].plot(ep, np.array(hist['val_perm'])*100, label='val')
    ax[2].axhline(100/Config.NUM_PERMUTATIONS, ls='--', c='grey', label='chance')
    ax[2].set_title('Permutation accuracy (%)'); ax[2].legend(); ax[2].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=140); plt.close(fig)
    print(f"✓ curves → {path}")


def run(check_only=False, smoke=False):
    if smoke:
        Config.EPOCHS = Config.SMOKE_EPOCHS
        Config.TRAIN_SAMPLES_PER_EPOCH = Config.SMOKE_TRAIN
        Config.VAL_SAMPLES = Config.SMOKE_VAL
        Config.RESUME = False
    set_seed(Config.SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("=" * 70)
    print("TISSUE RECONSTRUCTION PRETRAINING — Swin-Tiny")
    print("=" * 70)
    print(f"device       : {device}")
    print(f"backbone     : {Config.BACKBONE} @ {get_input_size(Config.BACKBONE)}px")
    print(f"batch        : {Config.BATCH_SIZE} × accum {Config.GRAD_ACCUM_STEPS} = {Config.BATCH_SIZE * Config.GRAD_ACCUM_STEPS} effective")
    print(f"samples/epoch: {Config.TRAIN_SAMPLES_PER_EPOCH:,}")
    print(f"epochs       : {Config.EPOCHS}")
    print(f"mode         : {'SMOKE' if smoke else 'full'}")
    print("=" * 70)
    torch.cuda.empty_cache(); gc.collect()
    hf = load_full_dataset()
    if not check_schema(hf):
        sys.exit(1)
    index = SpatialIndex(hf, Config.GRID_SIZE,
                         cache_path=f"{Config.CACHE_DIR}/v2_swin_tiny_lattices.pkl")
    if check_only:
        print("\n✓ checks done.")
        return
    train_loader, val_loader = create_dataloaders(hf, index, Config)
    torch.cuda.empty_cache(); gc.collect()
    model = ReconstructionModel(
        Config.BACKBONE, Config.NUM_POSITIONS, Config.NUM_PERMUTATIONS,
        Config.PRETRAINED, Config.DROPOUT,
        Config.FREEZE_STAGES, Config.CONTEXT_LAYERS).to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"\n✓ {Config.BACKBONE}: dim={model.feature_dim}, {trainable:,}/{total:,} trainable")
    probe = next(iter(train_loader))
    with torch.no_grad():
        pl, ol = model(probe['images'][:2].to(device), probe['mask'][:2].to(device))
    print(f"✓ forward OK — perm {tuple(pl.shape)}, pos {tuple(ol.shape)}")
    del probe, pl, ol
    torch.cuda.empty_cache()
    trainer = Trainer(model, device, Config, steps_per_epoch=len(train_loader))
    hist = trainer.fit(train_loader, val_loader)
    out = f"{Config.CHECKPOINT_DIR}/v2_swin_tiny_final.pt"
    torch.save({'model_state_dict': model.state_dict(),
                'backbone': Config.BACKBONE, 'feature_dim': model.feature_dim,
                'history': dict(hist), 'best_pos_acc': trainer.best_pos}, out)
    print(f"\n✓ model → {out}")
    plot_history(hist)
    ch = 100.0 / Config.NUM_POSITIONS
    best = trainer.best_pos * 100
    gap = (hist['train_pos'][-1] - hist['val_pos'][-1]) * 100 if hist['train_pos'] else 0
    print("\n" + "=" * 70)
    print(f"best val per-patch accuracy : {best:.2f}%   (chance {ch:.2f}%)")
    print(f"above chance                : {best - ch:+.2f} pp")
    print(f"final train-val gap         : {gap:.1f} pp")
    print("=" * 70)
    return model, trainer, hist


if __name__ == '__main__':
    args = set(sys.argv[1:])
    run(check_only='--check' in args, smoke='--smoke' in args)

TISSUE RECONSTRUCTION PRETRAINING — Swin-Tiny
device       : cuda
backbone     : swin_tiny @ 224px
batch        : 8 × accum 6 = 48 effective
samples/epoch: 10,000
epochs       : 60
mode         : full


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

...

  train  loss 2.0923   perm  91.37%   pos  54.09%
  val    loss 1.5441   perm  98.30%   pos  61.34%
  per-patch above chance: +50.23 pp   | gap: -7.3 pp
  ✓ new best per-patch 61.34%
Epoch 57/60   lr=2.93e-05


  train  loss 2.0623   perm  92.00%   pos  54.50%
  val    loss 1.5170   perm  98.65%   pos  61.39%
  per-patch above chance: +50.28 pp   | gap: -6.9 pp
  ✓ new best per-patch 61.39%
Epoch 58/60   lr=2.93e-05


  train  loss 2.0451   perm  92.61%   pos  54.95%
  val    loss 1.5398   perm  98.25%   pos  61.31%
  per-patch above chance: +50.20 pp   | gap: -6.4 pp
Epoch 59/60   lr=2.92e-05


  train  loss 2.0262   perm  92.84%   pos  55.39%
  val    loss 1.5089   perm  98.40%   pos  62.33%
  per-patch above chance: +51.22 pp   | gap: -6.9 pp
  ✓ new best per-patch 62.33%
Epoch 60/60   lr=2.92e-05


  train  loss 1.9962   perm  93.01%   pos  56.44%
  val    loss 1.5130   perm  98.60%   pos  62.49%
  per-patch above chance: +51.38 pp   | gap: -6.1 pp
  ✓ new best per-patch 62.49%

✓ model → /kaggle/working/checkpoints/v2_swin_tiny_final.pt
✓ curves → /kaggle/working/results/v2_swin_tiny_curves.png

best val per-patch accuracy : 62.49%   (chance 11.11%)
above chance                : +51.38 pp
final train-val gap         : -6.1 pp
